# Bài toán 1 — Khám phá mối quan hệ giữa vị trí, cấp bậc kinh nghiệm và kỹ năng chuyên môn

Bản gốc tìm luật `ROLE → skill`. Bản này mở rộng thêm **cấp bậc kinh nghiệm (EXP)** lấy từ cột `experience_required`, chia thành 4 nhóm: `Intern`, `Fresher`, `Junior`, `Senior`. Câu hỏi nghiên cứu:

> Khi biết thêm cấp bậc kinh nghiệm của tin tuyển dụng, luật `{ROLE, EXP} → skill` có xếp hạng kỹ năng tốt hơn luật `ROLE → skill` hay không? Và so với cách đếm tần suất đơn giản thì sao?

Sáu mô hình được so sánh bằng kiểm định chéo phân tầng 5-fold lặp lại, kèm so sánh cặp (paired) theo từng fold.

## 1. Dữ liệu và thiết kế đánh giá

Loại nhóm role `Other` và các tin không trích xuất được kỹ năng chuyên môn. Ngoài các cột `ROLE_*` và kỹ năng, ma trận giao dịch giờ có thêm các cột `EXP_*` (tạo ở notebook `EDA_and_Preprocessing`).

Quy ước cấp bậc (suy từ `experience_required`; chi tiết ở notebook EDA):

| Cấp bậc | Quy tắc |
|---|---|
| Intern | Không yêu cầu kinh nghiệm |
| Fresher | Trên 0 đến 12 tháng |
| Junior | Trên 1 năm đến 3 năm |
| Senior | Trên 3 năm |

`Intern` ở đây nghĩa là tin **không đòi hỏi kinh nghiệm** (gồm cả thực tập sinh lẫn vị trí nhận người mới), không phải nhãn lấy từ chức danh.

In [ ]:
import ast, json, math, time, tracemalloc
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from scipy.stats import wilcoxon
from sklearn.model_selection import RepeatedStratifiedKFold

DATA_DIR = Path("../data/processed")
RESULT_DIR = Path("../result/problem_1")
FIGURES_DIR = RESULT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE, N_SPLITS, N_REPEATS = 42, 5, 3   # 5 fold x 3 lần lặp = 15 lần đánh giá (đặt N_REPEATS=1 để chạy nhanh)
MIN_SUPPORT_COUNT, MIN_CONFIDENCE, MIN_LIFT = 8, 0.20, 1.20
MIN_SUPPORT_COUNT_EXP = 5   # mỗi ô ROLE x EXP nhỏ hơn cả role nên giảm ngưỡng support tuyệt đối
MAX_ITEMSET_LENGTH = 2      # 1 ngữ cảnh (ROLE hoặc ROLE x EXP) + 1 skill
SMOOTH_ALPHA, MIN_CELL_JOBS = 20, 10   # tham số baseline "Phổ biến theo role + EXP"
EXP_LEVELS = ["Intern", "Fresher", "Junior", "Senior"]
TOP_K_VALUES = [5, 10]

def parse_skill_list(value):
    if isinstance(value, list): parsed = value
    elif isinstance(value, str):
        try: parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError): parsed = []
    else: parsed = []
    if not isinstance(parsed, list): return []
    return list(dict.fromkeys(x.strip() for x in parsed if isinstance(x, str) and x.strip()))

jobs_all = pd.read_csv(DATA_DIR / "processed_jobs.csv")
jobs_all["skills_final"] = jobs_all["skills_final"].apply(parse_skill_list)
if "exp_level" not in jobs_all:
    raise ValueError("Thiếu cột exp_level. Hãy chạy lại notebook EDA_and_Preprocessing trước.")
jobs_all["exp_level"] = jobs_all["exp_level"].fillna("Unknown")
role_mask = jobs_all["role_group"].fillna("Other").ne("Other")
jobs = jobs_all.loc[role_mask].reset_index(drop=True)
transactions = pd.read_csv(DATA_DIR / "transaction_matrix.csv").replace(
    {"True": True, "False": False}
).fillna(False).astype(bool).reset_index(drop=True)
if len(jobs) != len(transactions):
    raise ValueError("processed_jobs và transaction_matrix không khớp thứ tự dòng sau khi bỏ Other")
role_cols = [c for c in transactions if c.startswith("ROLE_")]
exp_cols = [c for c in transactions if c.startswith("EXP_")]
skill_cols = [c for c in transactions if not c.startswith(("ROLE_", "EXP_"))]
if not exp_cols:
    raise ValueError("Ma trận giao dịch chưa có cột EXP_*. Hãy chạy lại notebook EDA_and_Preprocessing.")

role_from_matrix = transactions[role_cols].idxmax(axis=1).str.replace("ROLE_", "", n=1, regex=False)
if not role_from_matrix.equals(jobs["role_group"].reset_index(drop=True)):
    raise ValueError("Nhãn role không khớp thứ tự dòng của ma trận giao dịch")
exp_from_matrix = pd.Series("Unknown", index=transactions.index)
has_exp = transactions[exp_cols].any(axis=1)
exp_from_matrix[has_exp] = transactions.loc[has_exp, exp_cols].idxmax(axis=1).str.replace("EXP_", "", n=1, regex=False)
if not exp_from_matrix.equals(jobs["exp_level"].reset_index(drop=True)):
    raise ValueError("Nhãn exp_level không khớp ma trận giao dịch")

has_skills = jobs["skills_final"].map(bool)
excluded_empty = int((~has_skills).sum())
jobs = jobs.loc[has_skills].reset_index(drop=True)
transactions = transactions.loc[has_skills].reset_index(drop=True)

# Ngữ cảnh kết hợp ROLE x EXP: mỗi cột = 1 ô (role, cấp bậc). Luật {ROLE, EXP} -> skill
# tương đương luật "ô ROLE x EXP" -> skill với max_len=2.
ctx_matrix = pd.DataFrame({f"{r}|{e}": transactions[r] & transactions[e]
                           for r in role_cols for e in exp_cols if (transactions[r] & transactions[e]).any()})
print(f"Tin dùng được: {len(jobs)}; loại do thiếu skill: {excluded_empty}; số role: {jobs.role_group.nunique()}; số ô ROLE x EXP: {ctx_matrix.shape[1]}")
display(pd.crosstab(jobs["role_group"], jobs["exp_level"]).reindex(columns=EXP_LEVELS, fill_value=0).assign(Tổng=lambda d: d.sum(axis=1)))

## 2. Khai phá và xếp hạng luật

Sáu mô hình:

| Mô hình | Cách xếp hạng skill cho một tin có (role, cấp bậc) |
|---|---|
| Apriori / FP-Growth | Luật `ROLE → skill`, xếp theo confidence, lift, support (bản gốc) |
| Phổ biến theo role | Đếm tần suất skill trong role (baseline gốc) |
| Apriori + EXP / FP-Growth + EXP | Luật `{ROLE, EXP} → skill` xếp trước, sau đó **backoff** sang luật `ROLE → skill` cho các skill còn lại |
| Phổ biến theo role + EXP | Tần suất trong ô (role, cấp bậc), làm mượt về tần suất của role: `(đếm_ô + α·p_role) / (n_ô + α)`; ô dưới `MIN_CELL_JOBS` tin thì dùng tần suất role |

Mô hình "Phổ biến theo role + EXP" là **đối chứng công bằng** cho hai mô hình luật + EXP: nó cho biết lợi ích đến từ chính thông tin kinh nghiệm hay từ cơ chế luật.

Mỗi luật `ROLE → skill` cần tối thiểu 8 tin trong train, luật `{ROLE, EXP} → skill` cần tối thiểu 5 tin (ô nhỏ hơn). Ngưỡng confidence và lift giữ nguyên cho cả hai.

In [ ]:
RULE_COLS = ["context", "role_group", "exp_level", "skill", "support", "support_count", "confidence", "lift"]

def build_rules(freq, n_rows):
    # mlxtend >= 0.23.4 yêu cầu num_itemsets; các bản cũ hơn không nhận tham số này.
    try:
        return association_rules(freq, num_itemsets=n_rows, metric="confidence", min_threshold=MIN_CONFIDENCE)
    except TypeError:
        return association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)

def mine_rules(matrix, method, context_cols, min_count):
    # Luật {context} -> skill. context là 1 cột: ROLE_x (luật gốc) hoặc ROLE_x|EXP_y (luật theo cấp bậc).
    miner = apriori if method == "Apriori" else fpgrowth
    ctx = set(context_cols)
    freq = miner(matrix, min_support=min_count / len(matrix),
                 use_colnames=True, max_len=MAX_ITEMSET_LENGTH)
    empty = pd.DataFrame(columns=RULE_COLS)
    if freq.empty or not freq["itemsets"].map(len).ge(2).any():
        return freq, empty
    rules = build_rules(freq, len(matrix))
    if rules.empty: return freq, empty
    rules = rules[
        rules.antecedents.map(lambda x: len(x) == 1 and next(iter(x)) in ctx)
        & rules.consequents.map(lambda x: len(x) == 1 and next(iter(x)) not in ctx)
        & (rules.lift >= MIN_LIFT)
        & (rules.support * len(matrix) >= min_count - 1e-9)
    ].copy()
    if rules.empty: return freq, empty
    rules["context"] = rules.antecedents.map(lambda x: next(iter(x)))
    rules["role_group"] = rules["context"].map(lambda c: c.partition("|")[0].replace("ROLE_", "", 1))
    rules["exp_level"] = rules["context"].map(lambda c: c.partition("|")[2].replace("EXP_", "", 1) or None)
    rules["skill"] = rules.consequents.map(lambda x: next(iter(x)))
    rules["support_count"] = (rules.support * len(matrix)).round().astype(int)
    return freq, rules[RULE_COLS].sort_values(
        ["role_group", "exp_level", "confidence", "lift", "support"],
        ascending=[True, True, False, False, False]).reset_index(drop=True)

def rule_rankings(rules):
    # context -> danh sách skill xếp theo confidence, lift, support
    return {ctx: g.sort_values(["confidence", "lift", "support"], ascending=False).skill.drop_duplicates().tolist()
            for ctx, g in rules.groupby("context")}

def popularity_rankings(train_jobs, train_matrix):
    # Trả về (xếp hạng theo role, xếp hạng theo role+EXP có làm mượt và backoff).
    X = train_matrix[skill_cols].to_numpy(dtype=float)
    roles, exps = train_jobs.role_group.to_numpy(), train_jobs.exp_level.to_numpy()
    order = lambda scores: [skill_cols[i] for i in np.argsort(-scores, kind="stable")]
    role_rank, cell_rank = {}, {}
    for role in np.unique(roles):
        in_role = roles == role
        role_sum = X[in_role].sum(axis=0)
        role_rank[f"ROLE_{role}"] = order(role_sum)          # đếm thô, giống baseline gốc
        prior = role_sum / in_role.sum()
        for exp in EXP_LEVELS:
            in_cell = in_role & (exps == exp)
            if in_cell.sum() >= MIN_CELL_JOBS:
                score = (X[in_cell].sum(axis=0) + SMOOTH_ALPHA * prior) / (in_cell.sum() + SMOOTH_ALPHA)
                cell_rank[f"ROLE_{role}|EXP_{exp}"] = order(score)
    return role_rank, cell_rank

def predict(cell_rank, role_rank, role, exp):
    # Ưu tiên danh sách theo ô (role, EXP); thiếu thì backoff sang danh sách theo role.
    first = cell_rank.get(f"ROLE_{role}|{'EXP_' + exp}", [])
    return list(dict.fromkeys(first + role_rank.get(f"ROLE_{role}", [])))

def metrics_at_k(prediction, relevant, k):
    relevant = set(relevant) & set(skill_cols)
    prediction = list(dict.fromkeys(prediction))[:k]
    hits = [int(x in relevant) for x in prediction]
    dcg = sum(hit / math.log2(i + 2) for i, hit in enumerate(hits))
    idcg = sum(1 / math.log2(i + 2) for i in range(min(k, len(relevant))))
    return {"Precision@K": sum(hits) / k,
            "Recall@K": sum(hits) / len(relevant) if relevant else 0.0,
            "Hit@K": float(any(hits)), "NDCG@K": dcg / idcg if idcg else 0.0,
            "Recommendation coverage": len(prediction) / k}

## 3. Đánh giá phân tầng 5-fold lặp lại

Mỗi mô hình được huấn luyện trên bốn fold và đánh giá trên fold còn lại; lặp lại `N_REPEATS` lần với cách chia khác nhau để có nhiều quan sát cho so sánh cặp. Khi đánh giá, cấp bậc kinh nghiệm của tin kiểm tra được dùng làm đầu vào (nó là thông tin biết trước của tin tuyển dụng), còn danh sách skill thật chỉ dùng để chấm điểm.

`Precision@K` đo tỷ lệ gợi ý đúng trong K vị trí; `Recall@K` đo phần kỹ năng mục tiêu tìm được; `Hit@K` cho biết có ít nhất một gợi ý đúng hay không; `NDCG@K` đánh giá thứ tự xếp hạng. Chi phí của mô hình "+ EXP" gồm cả lần khai phá luật theo role (để backoff) lẫn lần khai phá luật theo ô role × EXP.

In [ ]:
MODELS = ["Apriori", "FP-Growth", "Phổ biến theo role",
          "Apriori + EXP", "FP-Growth + EXP", "Phổ biến theo role + EXP"]
metric_names = ["Precision@K", "Recall@K", "Hit@K", "NDCG@K", "Recommendation coverage"]
splitter = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
fold_quality, fold_role_quality, fold_benchmarks, fold_itemsets_equal = [], [], [], []

def timed(fn):
    tracemalloc.start(); started = time.perf_counter()
    result = fn()
    runtime = time.perf_counter() - started
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return result, runtime, peak / 2**20

for fold, (tr, te) in enumerate(splitter.split(np.zeros(len(jobs)), jobs.role_group), 1):
    train_jobs, test_jobs = jobs.iloc[tr].reset_index(drop=True), jobs.iloc[te].reset_index(drop=True)
    train_matrix = transactions.iloc[tr].reset_index(drop=True)
    train_ctx = ctx_matrix.iloc[tr].reset_index(drop=True)
    role_matrix = train_matrix[role_cols + skill_cols]
    exp_matrix = pd.concat([train_ctx, train_matrix[skill_cols]], axis=1)
    models, itemset_maps = {}, {}
    for method in ["Apriori", "FP-Growth"]:
        (itemsets, rules), rt_role, mem_role = timed(
            lambda: mine_rules(role_matrix, method, role_cols, MIN_SUPPORT_COUNT))
        role_rank = rule_rankings(rules)
        models[method] = ({}, role_rank)
        fold_benchmarks.append({"fold": fold, "Model": method, "Runtime (s)": rt_role,
            "Memory Usage (MB)": mem_role, "Number of Rules": len(rules)})
        (itemsets_e, rules_e), rt_exp, mem_exp = timed(
            lambda: mine_rules(exp_matrix, method, list(train_ctx.columns), MIN_SUPPORT_COUNT_EXP))
        models[f"{method} + EXP"] = (rule_rankings(rules_e), role_rank)
        fold_benchmarks.append({"fold": fold, "Model": f"{method} + EXP", "Runtime (s)": rt_role + rt_exp,
            "Memory Usage (MB)": max(mem_role, mem_exp), "Number of Rules": len(rules) + len(rules_e)})
        itemset_maps[method] = ({frozenset(x) for x in itemsets["itemsets"]},
                                {frozenset(x) for x in itemsets_e["itemsets"]})
    fold_itemsets_equal.append(itemset_maps["Apriori"] == itemset_maps["FP-Growth"])
    (pop_role, pop_cell), rt, mem = timed(lambda: popularity_rankings(train_jobs, train_matrix))
    models["Phổ biến theo role"] = ({}, pop_role)
    models["Phổ biến theo role + EXP"] = (pop_cell, pop_role)
    for name in ["Phổ biến theo role", "Phổ biến theo role + EXP"]:
        fold_benchmarks.append({"fold": fold, "Model": name, "Runtime (s)": rt,
            "Memory Usage (MB)": mem, "Number of Rules": np.nan})
    for name in MODELS:
        cell_rank, role_rank = models[name]
        predictions = [predict(cell_rank, role_rank, row.role_group, row.exp_level)
                       for row in test_jobs.itertuples(index=False)]
        for k in TOP_K_VALUES:
            row_scores = pd.DataFrame([metrics_at_k(p, row.skills_final, k)
                                       for p, row in zip(predictions, test_jobs.itertuples(index=False))])
            fold_quality.append({"fold": fold, "Model": name, "K": k,
                **row_scores[metric_names].mean().to_dict(), "Test jobs": len(row_scores)})
            by_role = row_scores.assign(role_group=test_jobs.role_group.to_numpy()).groupby("role_group")[metric_names].mean()
            for role, vals in by_role.iterrows():
                fold_role_quality.append({"fold": fold, "Model": name, "K": k, "role_group": role, **vals.to_dict()})

fold_quality_df = pd.DataFrame(fold_quality)
quality_df = fold_quality_df.groupby(["Model", "K"])[metric_names].agg(["mean", "std"]).reset_index()
quality_df.columns = ["Model", "K", *[f"{m}_{s}" for m in metric_names for s in ["mean", "std"]]]
quality_df["Model"] = pd.Categorical(quality_df["Model"], MODELS, ordered=True)
quality_df = quality_df.sort_values(["K", "Model"]).reset_index(drop=True)
quality_df["Model"] = quality_df["Model"].astype(str)
quality_df["folds"] = N_SPLITS * N_REPEATS
role_quality_df = (pd.DataFrame(fold_role_quality).groupby(["Model", "K", "role_group"])[metric_names]
                   .mean().reset_index())
bench_folds = pd.DataFrame(fold_benchmarks)
benchmark_df = bench_folds.groupby("Model").agg(
    **{"Runtime median (s)": ("Runtime (s)", "median"), "Runtime std (s)": ("Runtime (s)", "std"),
       "Number of Rules mean": ("Number of Rules", "mean"), "Number of Rules std": ("Number of Rules", "std"),
       "Memory Usage median (MB)": ("Memory Usage (MB)", "median"),
       "Memory Usage std (MB)": ("Memory Usage (MB)", "std")}).reindex(MODELS).reset_index()
print("Itemsets Apriori/FP-Growth giống nhau ở mọi fold:", all(fold_itemsets_equal))
display(quality_df.rename(columns={"Model": "Mô hình"}).round(4))
display(benchmark_df.rename(columns={"Model": "Mô hình", "Runtime median (s)": "Runtime trung vị (giây)",
    "Runtime std (s)": "Độ lệch chuẩn runtime", "Number of Rules mean": "Số luật trung bình",
    "Number of Rules std": "Độ lệch chuẩn số luật", "Memory Usage median (MB)": "Bộ nhớ Python trung vị (MB)",
    "Memory Usage std (MB)": "Độ lệch chuẩn bộ nhớ"}).round(4))

### 3.1 So sánh cặp theo từng fold

Với mỗi cặp mô hình, tính hiệu số điểm theo từng fold: `Δ = mô hình đầu − mô hình đối chứng`. Cột *fold tốt hơn* đếm số fold mà mô hình đầu thắng. Kiểm định Wilcoxon chỉ mang tính tham khảo vì các fold lặp lại dùng chung dữ liệu nên **không độc lập**; p-value có xu hướng quá lạc quan. Một cải thiện đáng tin khi Δ cùng dấu ở hầu hết các fold, chứ không chỉ khi p nhỏ.

In [ ]:
PAIRS = [("Apriori + EXP", "Apriori"), ("FP-Growth + EXP", "FP-Growth"),
         ("Phổ biến theo role + EXP", "Phổ biến theo role"),
         ("Apriori + EXP", "Phổ biến theo role + EXP"), ("Apriori + EXP", "Phổ biến theo role")]
pair_rows = []
for name, ref in PAIRS:
    for k in TOP_K_VALUES:
        a = fold_quality_df[(fold_quality_df.Model == name) & (fold_quality_df.K == k)].set_index("fold")
        b = fold_quality_df[(fold_quality_df.Model == ref) & (fold_quality_df.K == k)].set_index("fold")
        for metric in ["Precision@K", "Recall@K", "Hit@K", "NDCG@K"]:
            d = (a[metric] - b[metric]).dropna()
            try: p_value = wilcoxon(d).pvalue if (d != 0).any() else 1.0
            except ValueError: p_value = np.nan
            pair_rows.append({"So sánh": f"{name} − {ref}", "K": k, "Metric": metric,
                "Δ trung bình": d.mean(), "Δ std": d.std(), "Fold tốt hơn": int((d > 0).sum()),
                "Fold kém hơn": int((d < 0).sum()), "Tổng fold": len(d), "p (Wilcoxon, tham khảo)": p_value})
paired_df = pd.DataFrame(pair_rows)
display(paired_df.round(4))

### 3.2 Kết quả theo từng role

Điểm trung bình toàn cục có thể bị các role lớn (Designer, BA/PM, Backend) chi phối. Bảng dưới cho thấy NDCG@5 theo role của baseline và các mô hình có kinh nghiệm; chênh lệch dương nghĩa là thêm EXP có ích cho role đó.

In [ ]:
by_role = role_quality_df[role_quality_df.K == 5].pivot(index="role_group", columns="Model", values="NDCG@K")
by_role = by_role[MODELS]
by_role["Δ Phổ biến + EXP − Phổ biến"] = by_role["Phổ biến theo role + EXP"] - by_role["Phổ biến theo role"]
by_role["Δ Apriori + EXP − Apriori"] = by_role["Apriori + EXP"] - by_role["Apriori"]
by_role.insert(0, "Số tin", jobs.role_group.value_counts().reindex(by_role.index))
display(by_role.round(4))

## 4. Luật cuối cùng và độ bao phủ

Các file luật được khai phá từ toàn bộ tin đủ điều kiện để tiện xem xét; điểm chất lượng ở trên chỉ được tính từ các fold giữ lại. Bảng theo role cho thấy role nào có ít hoặc không có luật, bảng theo ô role × cấp bậc cho thấy ô nào đủ dữ liệu để có luật riêng.

In [ ]:
role_matrix_full = transactions[role_cols + skill_cols]
exp_matrix_full = pd.concat([ctx_matrix, transactions[skill_cols]], axis=1)
full_apriori_itemsets, rules_apriori = mine_rules(role_matrix_full, "Apriori", role_cols, MIN_SUPPORT_COUNT)
full_fp_itemsets, rules_fpgrowth = mine_rules(role_matrix_full, "FP-Growth", role_cols, MIN_SUPPORT_COUNT)
full_apriori_exp, rules_apriori_exp = mine_rules(exp_matrix_full, "Apriori", list(ctx_matrix.columns), MIN_SUPPORT_COUNT_EXP)
full_fp_exp, rules_fpgrowth_exp = mine_rules(exp_matrix_full, "FP-Growth", list(ctx_matrix.columns), MIN_SUPPORT_COUNT_EXP)
same = lambda x, y: {frozenset(i) for i in x["itemsets"]} == {frozenset(i) for i in y["itemsets"]}
print("Itemsets Apriori = FP-Growth (luật ROLE):", same(full_apriori_itemsets, full_fp_itemsets),
      "| (luật ROLE x EXP):", same(full_apriori_exp, full_fp_exp))

coverage_rows = []
for role in sorted(jobs.role_group.unique()):
    a, f = rules_apriori[rules_apriori.role_group.eq(role)], rules_fpgrowth[rules_fpgrowth.role_group.eq(role)]
    ae = rules_apriori_exp[rules_apriori_exp.role_group.eq(role)]
    idx = jobs.role_group.eq(role).to_numpy()
    coverage_rows.append({"role_group": role, "usable_jobs": int(idx.sum()),
        "apriori_rules": len(a), "apriori_candidate_skills": a.skill.nunique(),
        "fpgrowth_rules": len(f), "fpgrowth_candidate_skills": f.skill.nunique(),
        "apriori_exp_rules": len(ae), "apriori_exp_candidate_skills": ae.skill.nunique(),
        "role_popularity_candidate_skills": int((transactions.loc[idx, skill_cols].sum() > 0).sum())})
role_coverage_df = pd.DataFrame(coverage_rows)
display(role_coverage_df.rename(columns={"role_group": "Nhóm role"}))

cell_rows = []
for role in sorted(jobs.role_group.unique()):
    for exp in EXP_LEVELS:
        idx = (jobs.role_group.eq(role) & jobs.exp_level.eq(exp)).to_numpy()
        ae = rules_apriori_exp[rules_apriori_exp.role_group.eq(role) & rules_apriori_exp.exp_level.eq(exp)]
        cell_rows.append({"role_group": role, "exp_level": exp, "usable_jobs": int(idx.sum()),
            "apriori_exp_rules": len(ae), "apriori_exp_candidate_skills": ae.skill.nunique(),
            "meets_popularity_min_jobs": bool(idx.sum() >= MIN_CELL_JOBS)})
role_exp_coverage_df = pd.DataFrame(cell_rows)
display(role_exp_coverage_df[role_exp_coverage_df.usable_jobs > 0].rename(columns={"role_group": "Nhóm role", "exp_level": "Cấp bậc"}))

### 4.1 Ví dụ luật theo cấp bậc

Hai luật có lift cao nhất trong mỗi ô role × cấp bậc (tối đa 40 dòng). Đây là phần diễn giải: cùng một role nhưng kỹ năng đặc trưng thay đổi theo cấp bậc.

In [ ]:
example_rules = (rules_apriori_exp.sort_values(["role_group", "exp_level", "lift"], ascending=[True, True, False])
                 .groupby(["role_group", "exp_level"]).head(2).reset_index(drop=True))
display(example_rules[["role_group", "exp_level", "skill", "support_count", "confidence", "lift"]].head(40).round(3))

## 5. Biểu đồ và file kết quả

Các file được lưu trong `result/problem_1/`. Bộ nhớ là lượng cấp phát đỉnh của Python đo bằng `tracemalloc`, không bao gồm toàn bộ phần bộ nhớ native do NumPy/pandas sử dụng.

In [ ]:
PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974", "#64B5CD"]
plot_df = quality_df.copy()
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
for ax, metric, vn_title in zip(axes, ["Precision@K", "Recall@K", "Hit@K", "NDCG@K"],
    ["Precision@K — Độ chính xác", "Recall@K — Độ bao phủ", "Hit@K — Tỷ lệ trúng", "NDCG@K — Thứ hạng"]):
    pivot = plot_df.pivot(index="Model", columns="K", values=f"{metric}_mean").reindex(MODELS)
    pivot.plot(kind="bar", ax=ax, ylim=(0, 1), rot=35)
    ax.set_title(vn_title); ax.set_ylabel(f"Trung bình qua {N_SPLITS * N_REPEATS} lần đánh giá"); ax.set_xlabel("")
    ax.grid(axis="y", linestyle="--", alpha=.35); ax.legend(title="K")
    for label in ax.get_xticklabels(): label.set_ha("right")
fig.suptitle("Bài toán 1 — Chất lượng gợi ý (có và không có cấp bậc kinh nghiệm)"); fig.tight_layout()
fig.savefig(FIGURES_DIR / "recommendation_quality.png", dpi=180, bbox_inches="tight"); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, col, title in zip(axes, ["Runtime median (s)", "Number of Rules mean", "Memory Usage median (MB)"],
                          ["Thời gian chạy", "Số luật trung bình", "Bộ nhớ Python cực đại"]):
    ax.bar(benchmark_df.Model, benchmark_df[col], color=PALETTE)
    ax.set_title(title); ax.tick_params(axis="x", rotation=35); ax.grid(axis="y", linestyle="--", alpha=.35)
    for label in ax.get_xticklabels(): label.set_ha("right")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "benchmark.png", dpi=180, bbox_inches="tight"); plt.show()
fold_quality_df.to_csv(RESULT_DIR / "quality_metrics_by_fold.csv", index=False, encoding="utf-8-sig")
quality_df.to_csv(RESULT_DIR / "quality_metrics.csv", index=False, encoding="utf-8-sig")
role_quality_df.to_csv(RESULT_DIR / "quality_by_role.csv", index=False, encoding="utf-8-sig")
paired_df.to_csv(RESULT_DIR / "paired_comparison.csv", index=False, encoding="utf-8-sig")
benchmark_df.to_csv(RESULT_DIR / "benchmark.csv", index=False, encoding="utf-8-sig")
role_coverage_df.to_csv(RESULT_DIR / "role_coverage.csv", index=False, encoding="utf-8-sig")
role_exp_coverage_df.to_csv(RESULT_DIR / "role_exp_coverage.csv", index=False, encoding="utf-8-sig")
rules_apriori.to_csv(RESULT_DIR / "rules_apriori.csv", index=False, encoding="utf-8-sig")
rules_fpgrowth.to_csv(RESULT_DIR / "rules_fpgrowth.csv", index=False, encoding="utf-8-sig")
rules_apriori_exp.to_csv(RESULT_DIR / "rules_apriori_exp.csv", index=False, encoding="utf-8-sig")
rules_fpgrowth_exp.to_csv(RESULT_DIR / "rules_fpgrowth_exp.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả tại:", RESULT_DIR.resolve())

## 6. Diễn giải

Cách đọc kết quả, theo thứ tự:

1. **Luật + EXP so với luật gốc** (`Apriori + EXP − Apriori`, `FP-Growth + EXP − FP-Growth`): cho biết thông tin cấp bậc có làm luật tốt hơn không. Cải thiện đáng tin khi `Δ` dương ở hầu hết các fold.
2. **Luật + EXP so với đối chứng tần suất** (`Phổ biến theo role + EXP`): nếu baseline tần suất có EXP vẫn thắng luật + EXP, phần lớn lợi ích đến từ chính thông tin kinh nghiệm chứ không phải từ cơ chế luật, và bước lọc support/confidence/lift vẫn làm mất candidate hữu ích như ở bản gốc.
3. **Theo role** (mục 3.2): thêm EXP chỉ có thể giúp ở role có đủ tin trong nhiều cấp bậc. Với role chỉ có vài chục tin (Game Dev, Mobile Dev, Frontend), các ô role × cấp bậc dưới ngưỡng nên mô hình tự quay về luật theo role (backoff), và `Δ` gần 0 là điều dự kiến.
4. **Hạn chế dữ liệu**: nhãn `Intern` nghĩa là "không yêu cầu kinh nghiệm" chứ không phải nhãn thực tập sinh; skill đích được trích từ chính `technical_skills` và `requirements_text`, nên baseline tần suất vốn đã mạnh.

Dù kết quả thế nào, các luật theo cấp bậc (mục 4.1) vẫn hữu ích để diễn giải support, confidence và lift. Bài toán 2 tiếp tục kiểm tra ngữ cảnh cá nhân hóa `{role, cấp bậc, known skills} → next skill`.